## Session 5, in class: Markov chains with QuantEcon, time iteration, and the endogenous grid method

Computational Macro (HWS 2026), University of Mannheim. Thursday, 08.10.2026.

Three parts, one notebook. Part 1 puts income on a grid with the two methods of the slides, Tauchen and
Rouwenhorst, using the implementations of the `QuantEcon` package, and checks what the chains deliver.
Parts 2 and 3 solve the CARA household of the slides with the two Euler equation methods, time iteration
and the endogenous grid method, and measure both against the exact policy.

**How to work with this notebook**

- Run the cells from top to bottom (`Shift + Enter`). After a restart of the kernel, start again at the top:
  later cells use what earlier cells define.
- There are **10 gaps**, marked `___`. The comment line `# Gap k: ...` directly above a gap says what goes in.
  Every gap is one line, and most are one line of the slides.
- Cells without a gap run as they are. Read them anyway: they are part of the method, and the comments
  explain each step.
- Each part ends with a **Check**: a number to compare with. Do not move on before it fits.
- **Slides.** "Slide 12" and the like refer to the session 5 slides of Wednesday, by the number at the
  bottom of each slide. Keep them open next to the notebook: most gaps are one line of a slide.
- Stuck on one gap for more than five minutes? Reread the hint and the slide, ask your neighbour, or ask us.
  The solution notebook is posted after the session.

**Plan**

| Part | Content | Gaps | Slides | Time |
| --- | --- | --- | --- | --- |
| 1 | Income on a grid: Tauchen and Rouwenhorst with `QuantEcon`, moments, simulation | 1 to 4 | 2 to 7 | 30 min |
| 2 | Time iteration on the CARA household: expected marginal utility, the residual, bisection | 5 to 7 | 8 to 14, 21 | 30 min |
| 3 | The endogenous grid method | 8 to 10 | 17 to 20, 22 | 20 min |

**Notation, slides to code**

| Slides | Meaning | Code |
| --- | --- | --- |
| $\rho$, $\sigma_\varepsilon$, $\sigma_z = \sigma_\varepsilon / \sqrt{1 - \rho^2}$ | persistence, innovation and unconditional standard deviation of $\ln z$ | `rho`, `sig_eps`, `sig_z` |
| $\mathcal{Z}$, $\boldsymbol{\Pi}$, $\pi_{mn} = \Pr(z' = z_n \mid z = z_m)$ | income grid and transition matrix, rows sum to one | `zgrid`, `Pi` |
| $\boldsymbol{\lambda}$, $\boldsymbol{\lambda} = \boldsymbol{\Pi}^\top \boldsymbol{\lambda}$ | stationary distribution | `lam` |
| $\mathcal{A}$, $x_{im} = z_m + R a_i$ | asset grid and cash on hand | `gri.a`, `gri.x` |
| $\mathbf{c}_n$ | consumption policy on the grid, iterate $n$ | `C` |
| $\mathbf{EMU} = u_c(\mathbf{c}_n) \boldsymbol{\Pi}^\top$ | expected marginal utility tomorrow, at every $(a'_l, z_m)$ | `EMU` |
| $f_{im}(c) = u_c(c) - \beta R \, \widehat{EMU}_m(x_{im} - c)$ | the time iteration residual | `resid(c)` |
| $c^{\min}_{im}, c^{\max}_{im}$ | the bracket from the grid limits | `cmin`, `cmax` |
| $\tilde c_{lm}, \tilde a_{lm}$ | the endogenous grid of one EGM step | `ctilde`, `atilde` |
| $e_W$ | largest error against the exact policy on $a \in [0, 8]$ | `err_window` |

**Toolbox: the few Julia tools this notebook uses**

| Tool | What it does | Example |
| --- | --- | --- |
| `A * B` | matrix product (also row times column) | `lam' * zgrid` is $\sum_m \lambda_m z_m$ |
| `f.(x)`, `.+`, `.*`, `./` | the dot: apply elementwise | `exp.(x)`, `a .+ b` |
| `x'` | transpose: a column becomes a row | `a .+ z'` is the matrix of all sums $a_i + z_m$ |
| `sum(A, dims = 2)` | sum along each row | `sum(Pi, dims = 2)` gives one number per row |
| `ifelse.(cond, A, B)` | elementwise: `A` where `cond` is `true`, `B` elsewhere | |
| `interp1(xq, x, f)` | linear interpolation of the points `(x, f)` at the query points `xq` (defined in part 2) | |
| `searchsortedfirst(row, u)` | the first index `n` with `row[n] >= u` (for a sorted `row`) | |

Julia counts from 1: the first entry of `x` is `x[1]`, the last is `x[end]`. A policy is stored as an
$N_a \times N_z$ matrix: row `i` is the asset grid point $a_i$, column `m` the income state $z_m$.

### 0. Packages

New today: `QuantEcon`, the Julia package behind the QuantEcon lectures. It brings `tauchen`, `rouwenhorst`,
and a `MarkovChain` type with a simulator and the stationary distribution. The first cell activates the
course environment in `Julia/` and installs every package listed there that is missing on your laptop,
this week `QuantEcon` (a few minutes the first time, see the README for the steps if it fails).

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))   # the course environment in Julia/
Pkg.instantiate()                        # installs every package it lists that is missing on your laptop

In [ ]:
using Plots
using Printf
using LinearAlgebra
using Random
using Statistics
using QuantEcon                                          # new today: tauchen, rouwenhorst, MarkovChain

### 1. Income on a grid (gaps 1 to 4)

*Session 5, slides 2 to 7.*

Income is persistent in logs, $\ln z' = \rho \ln z + \sigma_\varepsilon \varepsilon'$ with
$\varepsilon' \sim N(0, 1)$, as in session 4. We replace this continuous process by a Markov chain:
$N_z$ grid points for $\ln z$ and a matrix $\boldsymbol{\Pi}$ of transition probabilities. A good chain
matches three numbers of the process: its mean $0$, its unconditional standard deviation
$\sigma_z = \sigma_\varepsilon / \sqrt{1 - \rho^2}$, and its autocorrelation $\rho$.

In [ ]:
# The income process of session 4: ln z' = ρ ln z + sig_eps ε',  ε' ~ N(0, 1)
rho = 0.9          # persistence
sig_eps = 0.2      # standard deviation of the innovation
nz = 7             # number of grid points for ln z, N_z

# The unconditional standard deviation of ln z: one of the three targets for the chain
sig_z = sig_eps / sqrt(1 - rho^2)               # unconditional sd of an AR(1): sig_eps / sqrt(1 - rho^2)
@printf("targets: mean 0, standard deviation %.4f, autocorrelation %g\n", sig_z, rho)

**Tauchen (1986), slide 4.** The function call

`mc = tauchen(N, ρ, σ, μ, n_std)`

builds the chain for
$y' = \mu + \rho y + \varepsilon'$ with $\varepsilon' \sim N(0, \sigma^2)$: `σ` is the standard deviation
of the **innovation**, `n_std` is the $m$ of the slides (the grid runs over $\pm m \sigma_z$), both `μ`
and `n_std` are positional. It returns a `MarkovChain` object with the transition matrix in `.p` and
the grid in `.state_values`.

In [ ]:
# Build the Tauchen chain for ln z (slide 4): N_z grid points and the N_z x N_z transition matrix.
# The result mc_t bundles both, the lines after the call print them.
# Gap 1: call tauchen with the number of states nz, the persistence rho, the innovation standard deviation
#        sig_eps, μ = 0.0, and n_std = 3 (the m of the slides), all positional
mc_t = ___

# mc_t is a MarkovChain object with two pieces:
#   mc_t.state_values   the grid for ln z, N_z points
#   mc_t.p              the transition matrix, N_z x N_z: row m holds the probabilities of tomorrow's states
#                       given state m today
println("grid for ln z: ", round.(mc_t.state_values, digits = 3))
println("transition matrix, rows are today, columns tomorrow:")
display(round.(mc_t.p, digits = 3))             # show the matrix rounded to 3 digits

**Rouwenhorst (1995), slides 5 and 6.** The function call

`mc = rouwenhorst(N, ρ, σ, μ)`

takes the same arguments as `tauchen` without `n_std` (`μ` is optional, default `0.0`): `σ` is again the
standard deviation of the **innovation**. The grid runs over $\pm \sqrt{N_z - 1} \, \sigma_z$, and the matrix
is built from the two state chain by padding, as on the slides. It returns a `MarkovChain` object, as
`tauchen` does.

In [ ]:
# Build the Rouwenhorst chain with the same arguments (slides 5 and 6), then check that every row of
# its transition matrix is a probability distribution.
# Gap 2: call rouwenhorst with the number of states, rho, and the innovation standard deviation
mc_r = ___
println("grid for ln z: ", round.(mc_r.state_values, digits = 3))
println("transition matrix:")
display(round.(mc_r.p, digits = 3))             # show the matrix rounded to 3 digits

In [ ]:
# Every row of p is a probability distribution over tomorrow's states, so every row must sum to one
# Gap 3: the sum of each row of mc_r.p, that is along dims = 2
rowsums = ___
println("rows sum to one: ", all(isapprox.(rowsums, 1)))

mid_state = nz ÷ 2 + 1                             # index of the middle state (Julia counts from 1)
println("probability to stay in the middle state: ", round(mc_r.p[mid_state, mid_state], digits = 3))

**What the chain delivers.** Three numbers, computed from the grid and the matrix alone.

- The stationary distribution $\boldsymbol{\lambda}$ solves $\boldsymbol{\lambda} = \boldsymbol{\Pi}^\top \boldsymbol{\lambda}$:
  it is the eigenvector of $\boldsymbol{\Pi}^\top$ to the eigenvalue one (slide 7). Here it is only a tool
  for the moments; next week is about distributions in their own right.
- Mean and variance weigh the grid with it: $\bar z = \sum_m \lambda_m z_m$ and
  $\text{Var}(z) = \sum_m \lambda_m (z_m - \bar z)^2$.
- The autocorrelation needs the probability of each pair (today, tomorrow): the pair $(z_m, z_n)$ has
  probability $\lambda_m \pi_{mn}$, so $\text{Cov}(z, z') = \sum_m \sum_n \lambda_m \pi_{mn} (z_m - \bar z)(z_n - \bar z)$,
  and the autocorrelation is $\text{Cov}(z, z') / \text{Var}(z)$.

In [ ]:
# The two functions below turn the formulas of the slides into code: the stationary distribution lam
# (slide 7), and the mean, standard deviation, and autocorrelation that Tauchen and Rouwenhorst are built
# to match (slides 3 to 6). The formulas are the three bullets above.
function stationary(Pi)
    # The stationary distribution lam of the chain: lam = Pi' lam, so lam is the eigenvector of Pi'
    # to the eigenvalue one, scaled to sum to one.
    F = eigen(Matrix(Pi'))                      # all eigenvalues (F.values) and eigenvectors (F.vectors) of Pi'
    k = argmin(abs.(F.values .- 1))             # position of the eigenvalue closest to one
    lam = real.(F.vectors[:, k])                # its eigenvector (eigen may return complex numbers, the imaginary part is 0)
    return lam ./ sum(lam)                      # an eigenvector has any scale: normalize it to a distribution
end

In [ ]:
# The moments of the chain, with the formulas of the bullets above: weigh the grid with lam.
function chain_moments(zgrid, Pi)
    # Mean, standard deviation, and autocorrelation of the chain (zgrid, Pi) in its stationary distribution.
    # (mean, var, std are functions from Statistics, so the mean is mz and the variance vz here.)
    lam = stationary(Pi)                        # the weights of the grid points in the long run
    mz = lam' * zgrid                           # sum_m lam_m z_m
    dev = zgrid .- mz                           # deviation from the mean, one per state
    vz = lam' * dev .^ 2                        # sum_m lam_m (z_m - mz)^2
    joint = lam .* Pi                           # joint[m, n] = lam_m Pi_mn, the probability of the pair (z_m, z_n)
    # Cov(z, z') = sum over all pairs (m, n) of joint[m, n] dev[m] dev[n]. The outer product dev * dev' is the
    # matrix of the products dev[m] dev[n]: multiply it elementwise with joint and add up all entries.
    cov = sum(joint .* (dev * dev'))
    return mz, sqrt(vz), cov / vz               # the autocorrelation is Cov(z, z') / Var(z)
end

In [ ]:
# Apply chain_moments to both chains and compare with the targets printed at the top of part 1.
for (name, mc) in [("Tauchen", mc_t), ("Rouwenhorst", mc_r)]
    mz, sd, ac = chain_moments(mc.state_values, mc.p)  # the three moments of this chain
    @printf("%-12s mean %+.4f  standard deviation %.4f  autocorrelation %.4f\n", name, mz, sd, ac)
end

Rouwenhorst hits both targets exactly, Tauchen does not: with $\rho = 0.9$ and seven points the
standard deviation is $0.54$ instead of $0.46$. The cell below repeats the comparison for three
persistences (nothing to fill in). Watch what happens to Tauchen's autocorrelation at $\rho = 0.99$, and
compare with Giovanni's presentation from yesterday.

In [ ]:
# Nothing to fill in. The same comparison for three persistences rho_ = 0.5, 0.9, 0.99. For each rho_:
#   1. compute the target standard deviation of the AR(1), sig_eps / sqrt(1 - rho_^2),
#   2. build both chains with this rho_ (the same calls as above),
#   3. compute their moments with chain_moments and print one row per method.
# Columns: persistence, method, standard deviation of the chain, its target, autocorrelation of the
# chain (its target is rho_ itself).
@printf("%5s %-12s %8s %8s %9s\n", "rho", "method", "sd", "target", "autocorr")
for rho_ in [0.5, 0.9, 0.99]
    target = sig_eps / sqrt(1 - rho_^2)         # the standard deviation the chain should have
    chains = [("Tauchen", tauchen(nz, rho_, sig_eps, 0.0, 3)),
              ("Rouwenhorst", rouwenhorst(nz, rho_, sig_eps))]
    for (name, mc) in chains    # the two chains for this rho_, one row each
        _, sd, ac = chain_moments(mc.state_values, mc.p)  # the mean is zero for both, so we skip it (_)
        @printf("%5.2f %-12s %8.4f %8.4f %9.4f\n", rho_, name, sd, target, ac)
    end
end

**Simulation.** One draw per period, as on slide 7: cumulate each row once, draw $u \sim U(0, 1)$,
and move to the first state $n$ whose cumulated probability reaches $u$. Example: if the row of today's
state is $(0.2, 0.5, 0.3)$, the cumulated row is $(0.2, 0.7, 1.0)$, and $u = 0.45$ moves the chain to the
second state. The function below does it by hand.
`QuantEcon` does the same in `simulate_indices(mc, T; init=...)` (`simulate` returns the grid values
instead of the indices). `Random.seed!` makes the draws reproducible: same seed, same path.

In [ ]:
# Simulation by hand (slide 7): cumulate each row once, then one uniform draw per period picks
# tomorrow's state from the cumulated row of today's state.
function simulate_chain(Pi, T, m0)
    # A simulated path of T state indices, starting in state m0
    F = cumsum(Pi, dims = 2)                    # cumulate each row once: F[m, n] = Pi[m, 1] + ... + Pi[m, n]
    s = Vector{Int}(undef, T)                   # s[t] is the index of the state in period t
    s[1] = m0                                   # the path starts in state m0
    for t in 1:T-1
        u = rand()                              # a fresh uniform draw on (0, 1)
        # Gap 4: the first n with F[s[t], n] >= u: searchsortedfirst(row, u) returns it, the row is today's
        #        cumulated row F[s[t], :]
        s[t+1] = ___
    end
    return s
end

In [ ]:
T = 100_000                                     # number of simulated periods (the _ only makes the number readable)
Random.seed!(11)                                        # fix the random numbers: same seed, same path
s = simulate_chain(mc_r.p, T, nz ÷ 2 + 1)               # start in the middle state
zpath = mc_r.state_values[s]                            # from state indices to values of ln z
autocorr(x) = cor(x[1:end-1], x[2:end])                 # correlation between x_t and x_{t+1}
@printf("by hand:    standard deviation %.4f, autocorrelation %.4f\n", std(zpath), autocorr(zpath))

In [ ]:
# The same with the simulator of QuantEcon
Random.seed!(11)                                # same seed as above, so QuantEcon draws the same random numbers
s_qe = simulate_indices(mc_r, T; init = nz ÷ 2 + 1)  # QuantEcon's simulator: T state indices, same start
zpath_qe = mc_r.state_values[s_qe]              # from state indices to values of ln z
@printf("QuantEcon:  standard deviation %.4f, autocorrelation %.4f\n", std(zpath_qe), autocorr(zpath_qe))

In [ ]:
# Nothing to fill in: the first 120 periods of the simulated path
p1 = plot(0:119, zpath[1:120], linetype = :steppost, title = "A simulated chain, 120 periods", xlabel = "period t", ylabel = "ln z", legend = false, size = (700, 300))
hline!(p1, mc_r.state_values, color = :gray, linestyle = :dot, linewidth = 0.5)   # the grid points as dotted lines

**Check:** by hand and with `QuantEcon`, the simulated standard deviation is about $0.46$ and the
autocorrelation about $0.90$. If your numbers are far off, look at gap 4: the cumulated row and the `searchsortedfirst`.

### 2. Time iteration on the CARA household (gaps 5 to 7)

*Session 5, slides 8 to 14, the benchmark on slide 21.*

The household of slides 8 and 9: $u(c) = -e^{-\gamma c} / \gamma$, $c + a' = z + R a$, i.i.d. income
$z = \bar z + e$ with $e$ on $N_z = 7$ Rouwenhorst points. With $\rho = 0$ all rows of
$\boldsymbol{\Pi}$ are identical, the row is the weight vector $\omega$. Parameters as on the slides:
$\gamma = 2$, $r = 0.04$, $\beta = 1/R$, $\bar z = 1$, $\sigma = 0.2$, $N_a = 200$ equidistant points
on $[-6, 14]$. The bounds of the asset grid act as a borrowing and a saving limit.

**The benchmark.** The exact policy of the discretized problem is $c = \mu x + f_{N_z}$ with $\mu = r / R$ and

$$
f_{N_z} = \frac{\mu \bar z}{r} - \frac{1}{\gamma r} \Bigl[ \ln(\beta R) + \ln \sum_m \omega_m \exp\bigl(-\gamma \mu (z_m - \bar z)\bigr) \Bigr],
$$

slide 21, "On the computer: an exact benchmark". Errors are measured on the window $a \in [0, 8]$, six
units inside each bound, where the limits do not reach.

**Time iteration in one sentence (slides 11 and 12).** Guess a policy $\mathbf{c}_n$, and at every grid point find the $c$ that
solves the Euler equation when tomorrow's consumption follows the guess. That gives $\mathbf{c}_{n+1}$.
Repeat until the policy stops changing. The next cells build this one step at a time.

The first two cells set up parameters, grids, and the benchmark. Nothing to fill in, but read the comments.

In [ ]:
Base.@kwdef struct EconomicParameters
    γ::Float64 = 2.0          # absolute risk aversion
    r::Float64 = 0.04         # net interest rate, R = 1 + r
    β::Float64 = 1 / 1.04     # discount factor, β R = 1
    zbar::Float64 = 1.0       # mean income
    σ::Float64 = 0.2          # unconditional standard deviation of income
    ρ::Float64 = 0.0          # persistence of income, 0 is i.i.d.
end

Base.@kwdef struct NumericalParameters          # the numerical choices: grid sizes, tolerances
    na::Int = 200             # asset grid points N_a
    nz::Int = 7               # income grid points N_z
    amin::Float64 = -6.0      # lower grid bound a_1
    amax::Float64 = 14.0      # upper grid bound a_{N_a}
    crit_pol::Float64 = 1e-6  # tolerance on the policy, both methods stop when the policy moves by less
    maxiter::Int = 5000       # iteration cap, the safety net
    nbisect::Int = 60         # bisection steps per time iteration step: bracket width 20 / 2^60
    window::Float64 = 6.0     # errors are reported on [a_1 + window, a_N - window]
end

In [ ]:
struct Grids                                    # a container for the grids, filled by make_grids
    a::Vector{Float64}        # asset grid, N_a
    z::Vector{Float64}        # income grid, N_z
    Pi::Matrix{Float64}       # transition matrix, N_z x N_z
    x::Matrix{Float64}        # cash on hand x_im = z_m + R a_i, N_a x N_z
end

R(par) = 1 + par.r            # gross interest rate, use it as R(par)

function make_grids(par, mpar)
    # Income z = zbar + e, with e on a Rouwenhorst grid. QuantEcon takes the standard deviation of the
    # innovation, sd(e) * sqrt(1 - ρ^2), so that the standard deviation of e itself is par.σ.
    sig_innov = par.σ * sqrt(1 - par.ρ^2)       # innovation sd that gives sd(e) = σ
    mc = rouwenhorst(mpar.nz, par.ρ, sig_innov) # the income chain, as in part 1
    a = collect(range(mpar.amin, mpar.amax, length = mpar.na))   # N_a equidistant asset grid points
    z = par.zbar .+ mc.state_values                               # income levels
    # Cash on hand x_im = z_m + R a_i for all combinations: a is a column (assets down the rows),
    # z' is a row (income along the columns), and their sum is an N_a x N_z matrix.
    x = z' .+ R(par) .* a                       # N_a x N_z: row i is a_i, column m is z_m
    return Grids(a, z, Matrix(mc.p), x)         # bundle everything in one container
end

In [ ]:
par, mpar = EconomicParameters(), NumericalParameters()  # the default values listed above
gri = make_grids(par, mpar)                     # grids, transition matrix, cash on hand
println("income grid: ", round.(gri.z, digits = 3))
println("weights:     ", round.(gri.Pi[1, :], digits = 4), "  all rows identical: ", all(isapprox.(gri.Pi, gri.Pi[1:1, :])))
println("shape of cash on hand: ", size(gri.x), " = (N_a, N_z)")

In [ ]:
# The dots inside make both functions work elementwise on a number, a vector, or a matrix
uprime(c, par) = exp.(-par.γ .* c)              # marginal utility of u(c) = -exp(-γ c) / γ
uprime_inv(m, par) = -log.(m) ./ par.γ          # inverse of marginal utility: the c with u'(c) = m (needed in part 3)

In [ ]:
function c_exact(par, gri)
    # The benchmark: the exact policy c = μ x + f of the discretized i.i.d. problem, from the slides
    Rp, r = R(par), par.r                       # short names for the formula (R is taken by the function R(par))
    μ = r / Rp                                  # marginal propensity to consume out of cash on hand
    ω = gri.Pi[1, :]                            # weights of the income states (all rows of Pi are the same)
    e = gri.z .- par.zbar                       # income shocks
    risk_term = log(sum(ω .* exp.(-par.γ * μ .* e)))  # the precautionary term of the formula
    f = μ * par.zbar / r - (log(par.β * Rp) + risk_term) / (par.γ * r)  # the intercept f_{N_z} of the slide
    return μ .* gri.x .+ f                      # c = μ x + f at every grid point
end

function err_window(C, par, mpar, gri)
    # Largest absolute consumption error against the exact policy: on the window, and on the whole grid
    err = abs.(C .- c_exact(par, gri))          # absolute error at every grid point
    inside = (gri.a .>= gri.a[1] + mpar.window) .& (gri.a .<= gri.a[end] - mpar.window)   # rows in the window
    return maximum(err[inside, :]), maximum(err)  # largest error on the window, and on the whole grid
end

In [ ]:
μ = par.r / R(par)                              # the marginal propensity to consume out of cash on hand
f = c_exact(par, gri)[1, 1] - μ * gri.x[1, 1]   # read the intercept off the exact policy
@printf("exact policy: mu = %.5f, intercept f = %.5f\n", μ, f)

The starting policy is the closed form without its precautionary term, $c_0 = \mu x + (1 - \mu) \bar z$,
the policy of a household that ignores the risk (the slides' $c = r a + \bar z$ works as well, it costs one
iteration more).

**Step 2 of the algorithm on slide 12: expected marginal utility (gap 5).** Given the guess $\mathbf{c}_n$ on the grid,
tomorrow's expected marginal utility at every $(a'_l, z_m)$ is one matrix product,
$\mathbf{EMU} = u_c(\mathbf{c}_n) \boldsymbol{\Pi}^\top$, exactly as $\mathbf{EV} = \mathbf{V} \boldsymbol{\Pi}^\top$ in week 4:
$EMU_{lm} = \sum_n \pi_{mn} \, u_c(c_n(a'_l, z_n))$. In words: a household with income $z_m$ today
draws tomorrow's income from row $m$ of $\boldsymbol{\Pi}$, and weighs tomorrow's marginal utilities with it.

In [ ]:
# Step 2 of time iteration (slide 12): from the guess C to expected marginal utility tomorrow, at every
# grid point for tomorrow's assets and every income state today.
function expected_mu(C, par, gri)
    # EMU[l, m] = Σ_n Pi[m, n] u'(C[l, n]): expected marginal utility tomorrow for a household that
    # saves a'_l and has income z_m today. Input C and output EMU are both N_a x N_z.
    # Gap 5: one matrix product, as EV = V Pi' in week 4, with the marginal utilities uprime(C, par) in place
    #        of V
    return ___
end

In [ ]:
# Test: with c = 1 everywhere, marginal utility is exp(-γ) everywhere, and so is its expectation
EMU = expected_mu(ones(size(gri.x)), par, gri)  # C = 1 at every grid point
println("expected marginal utility test: ", all(isapprox.(EMU, exp(-par.γ))))

**Step 3 on slide 12: the residual (gap 6).** For every $(a_i, z_m)$ and a candidate $c$, tomorrow's assets are
$a' = x_{im} - c$, in general off the grid, so column $m$ of $\mathbf{EMU}$ is interpolated linearly at $a'$:

$$
f_{im}(c) = u_c(c) - \beta R \, \widehat{EMU}_m(x_{im} - c).
$$

`interp1` is linear interpolation clamped to the grid (the `lin_interp` of week 3, for a vector of query points).
`resid` takes the whole $N_a \times N_z$ array of candidates at once and returns the array of residuals, one
interpolation per income state. It is decreasing in $c$ and has exactly one root per point:
$f > 0$ means $c$ is too low (marginal utility today is too high), $f < 0$ means $c$ is too high.

In [ ]:
function interp1(xq, xgrid, fgrid)
    # Linear interpolation of the points (xgrid, fgrid) at every query point in xq, clamped to the grid
    out = similar(xq)                           # an empty array of the same size as xq
    for (k, x) in enumerate(xq)
        i = clamp(searchsortedlast(xgrid, x), 1, length(xgrid) - 1)   # the interval [xgrid[i], xgrid[i+1]] around x
        w = (x - xgrid[i]) / (xgrid[i+1] - xgrid[i])                  # weight on the right end point
        out[k] = (1 - w) * fgrid[i] + w * fgrid[i+1]  # weighted average of the two neighbouring values
    end
    return out
end

In [ ]:
# Step 3 (slide 12): the Euler equation residual, in three moves: tomorrow's assets from the budget
# constraint, expected marginal utility at those assets (interpolated, gap 6), and u'(c) - beta R EMUhat.
function resid(c, EMU, par, gri)
    # The Euler equation residual f_im(c) = u'(c) - β R EMUhat_m(x_im - c), for an N_a x N_z array c
    aprime = gri.x .- c                         # tomorrow's assets implied by c, in general off the grid
    EMUhat = similar(c)                         # EMU interpolated at aprime, filled one column at a time
    for m in 1:length(gri.z)                    # one income state at a time
        # Gap 6: interpolate column m of EMU (values on the grid gri.a) at the choices aprime[:, m]:
        #        interp1(query points, grid, values)
        EMUhat[:, m] = ___
    end
    return uprime(c, par) .- par.β * R(par) .* EMUhat  # f = u'(c) - β R EMUhat: zero at the optimal c
end

In [ ]:
function guess(par, gri)
    # The starting policy c0 = μ x + (1 - μ) zbar: the closed form without the precautionary term.
    # min. caps it at x - a_1, the most a household can consume without borrowing beyond a_1.
    μ = par.r / R(par)                          # the marginal propensity to consume out of cash on hand
    c0 = μ .* gri.x .+ (1 - μ) * par.zbar       # consume the annuity value of cash on hand and income
    return min.(c0, gri.x .- gri.a[1])          # never consume so much that a' < a_1
end

In [ ]:
C0 = guess(par, gri)                            # the starting policy
f0 = resid(C0, expected_mu(C0, par, gri), par, gri)  # how far the guess is from solving the Euler equation
@printf("residual at the guess, largest absolute value: %.4f  (zero would mean the guess is the fixed point)\n", maximum(abs.(f0)))

**Steps 4 and 5: the limits, and bisection on all points at once (gap 7).** Bisection is on slides 13 and 14,
the bracket and the vectorized version on the backup slides "Bounds on consumption" and "Many points at once". The bracket comes from the grid,
$c^{\max} = x_{im} - a_1$ and $c^{\min} = x_{im} - a_{N_a}$. If $f_{im}(c^{\max}) \geq 0$ the household wants to
borrow beyond $a_1$ and the limit binds, if $f_{im}(c^{\min}) \leq 0$ it wants to save beyond $a_{N_a}$.
Otherwise the root lies inside, and bisection halves every bracket $60$ times: since $f$ is decreasing,
a positive residual at the midpoint means the root is to the right, so the lower end moves up to the
midpoint; otherwise the upper end moves down. The update of the lower end is given, gap 7 is the upper end.

In [ ]:
# Step 5 (slides 13 and 14, backup slide "Many points at once"): bisection on all grid points at once.
# Each pass halves every bracket [lo, hi]: evaluate f at the midpoint and keep the half with the root.
function bisect_vec(f, lo, hi, steps)
    # Bisection on every grid point at once (backup slide "Many points at once"): lo and hi are N_a x N_z arrays,
    # and the root of the decreasing function f lies between them, point by point.
    for _ in 1:steps
        mid = (lo .+ hi) ./ 2                   # midpoint of every bracket
        pos = f(mid) .> 0                       # true where f(mid) > 0: the root is in [mid, hi]
        lo = ifelse.(pos, mid, lo)              # where pos, the lower end moves up to mid, elsewhere it stays
        # Gap 7: the upper end: where pos it stays at hi, elsewhere (root in [lo, mid]) it moves down to mid.
        #        Mirror the line above
        hi = ___
    end
    return (lo .+ hi) ./ 2                      # after the last step, the midpoint is the root (up to 20 / 2^60)
end

In [ ]:
function time_iteration_step(C, par, mpar, gri)
    # One step of time iteration (the Coleman operator): the new policy from the guess C
    EMU = expected_mu(C, par, gri)              # step 2: tomorrow's expected marginal utility under the guess
    f = c -> resid(c, EMU, par, gri)            # step 3: the residual as a function of c alone

    # Step 4: the bracket from the grid limits, and where a limit binds
    cmax = gri.x .- gri.a[1]                    # consume everything down to a' = a_1
    cmin = gri.x .- gri.a[end]                  # save everything up to a' = a_N
    borrow = f(cmax) .>= 0                      # even cmax is too little: the borrowing limit binds
    save = f(cmin) .<= 0                        # even cmin is too much: the saving limit binds

    # Step 5: the root by bisection, then the limits where they bind
    c_root = bisect_vec(f, cmin, cmax, mpar.nbisect)  # the root of f in [cmin, cmax], at every grid point
    Cnew = ifelse.(save, cmin, c_root)          # where the saving limit binds, consume cmin
    Cnew = ifelse.(borrow, cmax, Cnew)          # where the borrowing limit binds, consume cmax
    return Cnew                                 # the new policy
end

In [ ]:
C1 = time_iteration_step(C0, par, mpar, gri)    # one step from the guess
interior = (C1 .< gri.x .- gri.a[1]) .& (C1 .> gri.x .- gri.a[end])   # points where neither limit binds
max_move = maximum(abs.(C1 .- C0))              # how much one step changes the policy
res = maximum(abs.(resid(C1, expected_mu(C0, par, gri), par, gri)[interior]))  # is C1 the root, given the EMU of the guess?
@printf("one step moves the guess by at most %.4f\n", max_move)
@printf("residual after the step, where no limit binds: %.1e  (machine precision: bisection found the roots)\n", res)

**Step 6: the loop.** Apply the step until the policy moves by less than $\varepsilon_{\text{pol}} = 10^{-6}$.
Nothing to fill in.

In [ ]:
function time_iteration(par, mpar, gri)
    # Iterate time_iteration_step until the policy moves by less than crit_pol
    t0 = time()                                 # start the clock
    C = guess(par, gri)                         # start from the guess
    dist = Inf                                  # distance between two iterates, infinite to enter the loop
    dists = Float64[]                           # all distances, to look at the convergence afterwards
    it = 0                                      # iteration counter
    while dist > mpar.crit_pol && it < mpar.maxiter
        Cnew = time_iteration_step(C, par, mpar, gri)  # one step of time iteration
        dist = maximum(abs.(Cnew .- C))         # largest change of the policy anywhere on the grid
        push!(dists, dist)                      # remember the distance
        C = Cnew                                # the new policy is the guess for the next step
        it += 1                                 # count the iteration
    end
    eW, eG = err_window(C, par, mpar, gri)      # errors against the exact policy
    return (C = C, iters = it, time = time() - t0, dist = dists,
            converged = dist <= mpar.crit_pol, eW = eW, eG = eG)
end

In [ ]:
ti = time_iteration(par, mpar, gri)             # solve the household problem with time iteration
@printf("time iteration: %d iterations, converged %s, %.2f s, %.1f ms per iteration, error on the window %.1e, on the grid %.1e\n",
        ti.iters, ti.converged, ti.time, 1000 * ti.time / ti.iters, ti.eW, ti.eG)

**Check:** 121 iterations and an error on the window of $6 \cdot 10^{-5}$, the numbers of slide 22,
"Four methods against the exact policy". The error on the whole grid is $0.47$: that is the limit at the
bottom of the grid, where the two lowest income states would like to borrow more, not a bug. The time per
iteration is what the endogenous grid method removes. If the loop does not converge, check gap 7 first:
with a wrong bracket update bisection finds no root.

### 3. The endogenous grid method (gaps 8 to 10)

*Session 5, slides 17 to 20.*

Carroll's trick (slide 17): fix tomorrow's assets $a'_l$ on the grid instead of today's. Then the right hand side of the
Euler equation is a number, $\beta R \, EMU_{lm}$, and consumption today follows by inverting marginal
utility, no root needed. The budget constraint then says where the household that chose $a'_l$ started:

$$
\tilde c_{lm} = u_c^{-1}\bigl( \beta R \, EMU_{lm} \bigr), \qquad \tilde a_{lm} = \frac{\tilde c_{lm} + a'_l - z_m}{R}.
$$

The points $(\tilde a_{lm}, \tilde c_{lm})$ lie on the new policy, on an endogenous grid that differs by
income state. Interpolating them back onto $\mathcal{A}$, one column at a time, gives $\mathbf{c}_{n+1}$ (slide 18).
As on slide 19, below $\tilde a_{1m}$ the borrowing limit binds and $c = x_{im} - a_1$, above $\tilde a_{N_a m}$ the saving
limit binds and $c = x_{im} - a_{N_a}$.

Compare with time iteration: steps 1 and 2 are the same (a guess and $\mathbf{EMU}$), only the step from
$\mathbf{EMU}$ to the new policy changes. The loop `egm` is the loop of part 2 with `egm_step` in place of
`time_iteration_step`.

In [ ]:
# One EGM step (algorithm on slide 20) in three moves: (a) consumption from the inverted Euler equation,
# (b) today's assets from the budget constraint (both slide 17), (c) back to the grid, with the limits
# where the endogenous grid ends (slides 18 and 19).
function egm_step(C, par, mpar, gri)
    # One step of the endogenous grid method: the new policy on the grid, and the endogenous grid of this step
    EMU = expected_mu(C, par, gri)              # as in time iteration: EMU[l, m] at tomorrow's assets a'_l and income z_m

    # (a) Consumption today from the Euler equation u'(ctilde) = β R EMU, no root finding needed
    # Gap 8: invert marginal utility with uprime_inv, applied to β R EMU (all points at once)
    ctilde = ___

    # (b) Where did the household start? The budget constraint c + a' = z + R a, solved for today's a
    aprime = gri.a                              # tomorrow's assets a'_l, a column with N_a entries
    z = gri.z'                                  # income z_m as a row, 1 x N_z
    # Gap 9: today's assets a = (c + a' - z) / R, with ctilde as c. Column plus row gives an N_a x N_z matrix,
    #        like ctilde (use the dots)
    atilde = ___

    # (c) Back to the grid, one income state at a time
    Cnew = similar(C)                           # the new policy on the grid, filled one column at a time
    for m in 1:length(gri.z)
        # Gap 10: the points (atilde[:, m], ctilde[:, m]) lie on the new policy of income state m: interpolate
        #         them at the grid points gri.a, interp1(query points, nodes, values)
        Cnew[:, m] = ___
        # interp1 is flat outside the endogenous grid, so overwrite those points with the limits
        below = gri.a .< atilde[1, m]           # poorer than the household that saves a' = a_1: borrowing limit binds
        above = gri.a .> atilde[end, m]         # richer than the household that saves a' = a_N: saving limit binds
        Cnew[below, m] = gri.x[below, m] .- gri.a[1]  # consume everything down to a' = a_1
        Cnew[above, m] = gri.x[above, m] .- gri.a[end]  # save everything up to a' = a_N
    end
    return Cnew, ctilde, atilde                 # the new policy, and the endogenous grid points
end

In [ ]:
function egm(par, mpar, gri)
    # The loop of time_iteration, with egm_step in place of time_iteration_step
    t0 = time()                                 # start the clock
    C = guess(par, gri)                         # start from the guess
    dist = Inf                                  # infinite to enter the loop
    dists = Float64[]                           # all distances
    it = 0                                      # iteration counter
    while dist > mpar.crit_pol && it < mpar.maxiter
        Cnew, _, _ = egm_step(C, par, mpar, gri)   # egm_step also returns the endogenous grid, not needed here
        dist = maximum(abs.(Cnew .- C))         # largest change of the policy
        push!(dists, dist)                      # remember the distance
        C = Cnew                                # the new policy is the guess for the next step
        it += 1                                 # count the iteration
    end
    eW, eG = err_window(C, par, mpar, gri)      # errors against the exact policy
    return (C = C, iters = it, time = time() - t0, dist = dists,
            converged = dist <= mpar.crit_pol, eW = eW, eG = eG)
end

In [ ]:
eg = egm(par, mpar, gri)                        # solve the household problem with the endogenous grid method
@printf("endogenous grid: %d iterations, converged %s, %.3f s, %.2f ms per iteration, error on the window %.1e, on the grid %.1e\n",
        eg.iters, eg.converged, eg.time, 1000 * eg.time / eg.iters, eg.eW, eg.eG)

In [ ]:
# One more step from the converged policy, to look at its endogenous grid
_, ctilde, atilde = egm_step(eg.C, par, mpar, gri)  # keep only the endogenous grid of this step
println("endogenous a at a' = a_1, by income state: ", round.(atilde[1, :], digits = 2))

**Check:** as on slide 22, 121 iterations, an error on the window of $2 \cdot 10^{-5}$, and the three lowest income
states have $\tilde a_{1m} > a_1 = -6$ (about $-5.7$, $-5.8$, $-6.0$), so the borrowing limit binds for them at
the bottom of the grid. The time per iteration should be an order of magnitude or two below time iteration.

In [ ]:
# Nothing to fill in. Left: one EGM step at the bottom of the grid for the lowest income state.
# Right: the error of both methods along the grid (largest over the income states), log scale.
ct = c_exact(par, gri)                          # the exact policy, for the comparison
sel = gri.a .<= -2.0                            # the bottom of the grid, where the limit binds
p1 = scatter(atilde[:, 1], ctilde[:, 1], markersize = 2.5, label = "EGM points (atilde, ctilde), lowest income",
             xlims = (gri.a[1] - 0.1, -2.0), ylims = (eg.C[1, 1] - 0.03, maximum(ct[sel, 1]) + 0.03),
             xlabel = "assets today a", ylabel = "consumption", title = "Back to the grid, and the limit", legend = :bottomright, titlefontsize = 10)
plot!(p1, gri.a[sel], eg.C[sel, 1], color = :black, linewidth = 1.2, label = "policy on the grid")
plot!(p1, gri.a[sel], ct[sel, 1], linestyle = :dot, linewidth = 1.6, label = "exact policy without the limit")
vline!(p1, [atilde[1, 1]], color = :gray, linestyle = :dash, label = "")   # below this point the borrowing limit binds
p2 = plot(xlabel = "assets a", ylabel = "max over m of |c - exact|", title = "Error against the exact policy (shaded: the window)", yscale = :log10, titlefontsize = 10)
for (name, res) in [("time iteration", ti), ("endogenous grid", eg)]
    err_by_a = vec(maximum(abs.(res.C .- ct), dims = 2))     # largest error over the income states, per grid point
    plot!(p2, gri.a, max.(err_by_a, 1e-16), label = name)
end
vspan!(p2, [0, 8], color = :gray, alpha = 0.15, label = "")
plot(p1, p2, layout = (1, 2), size = (950, 330))

In [ ]:
# The horse race of the slides, for the two Euler equation methods
@printf("%-18s %10s %13s %8s %9s %7s\n", "method", "iterations", "ms/iteration", "seconds", "e_W", "e_G")
for (name, res) in [("time iteration", ti), ("endogenous grid", eg)]
    @printf("%-18s %10d %13.2f %8.3f %9.1e %7.2f\n", name, res.iters, 1000 * res.time / res.iters, res.time, res.eW, res.eG)
end

**To discuss (slides 22 to 24).** Both methods iterate on the same operator, so they need the same number of
iterations (121 here, against 296 for value function iteration on the slides). The difference is the cost
per iteration: 60 bisection steps with 7 interpolations each, against 7 interpolations in total. The
remaining error of both methods on the window is neither stopping error nor interpolation error: tightening
`crit_pol` to $10^{-10}$ or doubling $N_a$ twice leaves the $2 \cdot 10^{-5}$ of the endogenous grid method
where it is (try it). It is the grid bounds: with $\sigma = 0.2$ the limits reach into the window, as the
summary on slide 24 says. Problem set 2 has $\sigma = 0.1$, where they do not.

(Julia: the first call of a function includes its compilation, so the times of the first run are too high.
Run the cells with `time_iteration` and `egm` once more, then this one, for the times to compare.)